# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all dataset elements by their Croissant `@id` values.

### Dataset Source
- Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- Title: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- DOI: 10.71728/senscience.vq4a-28xa


In [ ]:
# Ensure mlcroissant is installed (if running in Colab or similar environments; if local, you might already have it)
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and connect to the Croissant package via the schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Initialize and load the dataset
dataset = mlc.Dataset(croissant_url)

# Show dataset title and description
meta = dataset.metadata
print(f"{meta.name}\n{meta.description}")

## 2. Data Overview

List available record sets and their fields (using `@id`).

**Note:** All IDs are referenced by their `@id` in the Croissant schema, which ensures compatibility and reproducibility.

In [ ]:
# Examine available record sets, their @id, and fields

for rs in dataset.metadata.record_sets:
    print(f"RecordSet: {rs.name} | @id: {rs.id}")
    for field in rs.fields:
        print(f"  Field: {field.name} | @id: {field.id}")
    print('-' * 40)

# Example: list a few records from the main RecordSet
# Replace this with the actual record set @id you want to examine. We'll take the first one.
if len(dataset.metadata.record_sets) > 0:
    main_recordset_id = dataset.metadata.record_sets[0].id
    print(f"\nFirst 2 records for RecordSet @id: {main_recordset_id}")
    for i, rec in enumerate(dataset.records(record_set=main_recordset_id)):
        print(rec)
        if i >= 1:
            break

## 3. Data Extraction

Load all data from each RecordSet into a dictionary of pandas DataFrames. All entities are identified by their `@id` as required.

In [ ]:
# Collect all record set @id's
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]

# Load each record set into a DataFrame
dataframes = {}
for rset_id in record_sets_ids:
    records = list(dataset.records(record_set=rset_id))
    dataframes[rset_id] = pd.DataFrame(records)

if record_sets_ids:
    main_rs_id = record_sets_ids[0]  # we'll use the first RecordSet for analysis
    print(f"Columns for RecordSet @id '{main_rs_id}':")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()
else:
    print('No record sets found in the dataset.')

## 4. Exploratory Data Analysis (EDA)

We'll:  
- Choose a numeric field (via its `@id`)
- Filter the records where the value exceeds a threshold
- Normalize this field
- Optionally group by a categorical field (via its `@id`)

> **You can update the selected field IDs below to match a field relevant to your analysis. Default selections are made from the first RecordSet.

In [ ]:
main_rs = dataset.metadata.record_sets[0]
numeric_fields = [f for f in main_rs.fields if f.data_type in ["Float", "Number", "Integer"]]
group_fields = [f for f in main_rs.fields if f.data_type in ["Text", "String"]]

if numeric_fields:
    numeric_field_id = numeric_fields[0].id  # e.g., '@id' of some numeric value, e.g. 'cr:coverage_percent'
    print(f"Using numeric field: {numeric_field_id}")
else:
    print('No numeric field found.')
    numeric_field_id = None

if group_fields:
    group_field_id = group_fields[0].id  # e.g. '@id' of 'cr:description' or any categorical
    print(f"Grouping field: {group_field_id}")
else:
    group_field_id = None

df = dataframes[main_rs.id]

# Only run if numeric field is present and in DataFrame
if numeric_field_id and numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != 'O' else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records (first 5 rows):")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nAverage of {numeric_field_id} by {group_field_id} (first 5 groups):")
        print(grouped_df.head())
else:
    print("No valid numeric field available for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field.  
If a grouping field is available, plot the mean by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the selected numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# If grouped means were computed, plot them as well
if 'grouped_df' in locals() and not grouped_df.empty:
    grouped_df.head(20).plot(kind='bar', figsize=(8,4))
    plt.title(f"Average of {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

We have:
- Loaded the FAIR² mass spectrometry dataset via its Croissant schema using `mlcroissant`
- Listed all RecordSets and their field IDs (referenced by `@id`)
- Loaded the primary RecordSet into a DataFrame
- Demonstrated basic EDA: filtering, normalization, grouping
- Visualized the distribution of at least one numeric variable

**You can extend this notebook:**
- Analyze other RecordSets or fields by updating the selected `@id` variables
- Add advanced visualizations or machine learning workflows
- Explore croissant metadata for deeper semantic understanding and reproducibility
